In [18]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import cmocean
import os
from scipy.interpolate import griddata



filepath = r"C:\Users\marqjace\seaglider\sg266\sg266_2024_10_21_TH_Line_timeseries.nc"
figures_folder = r"C:\Users\marqjace\seaglider\sg266\figures"
os.makedirs(figures_folder, exist_ok=True)  # exist_ok=True prevents errors if the folder already exists

# Open the dataset
ds = xr.open_dataset(filepath)
# ds

# Variables
oxygen = ds.aanderaa4831_dissolved_oxygen
time_coverage_start = ds.attrs['time_coverage_start']
time_coverage_end = ds.attrs['time_coverage_end']

# Convert time variables to the same format
ctd_time = pd.to_datetime(ds.ctd_time.values, unit='s', origin='unix')
aa4831_time = pd.to_datetime(ds.aa4831_time.values, unit='s', origin='unix')

# Interpolate ctd_depth onto aa4831_time
aa4831_depth = np.interp(aa4831_time.astype(np.int64), ctd_time.astype(np.int64), ds.ctd_depth)

# Calculate and print the number of days the mission lasted
difference = aa4831_time.max() - aa4831_time.min()
num_days = difference.days
print(f'The mission lasted {num_days} days.') 

oxy_time_timestamps = aa4831_time.astype(np.int64) // 10**9

# Time vs Depth Grid (using the ctd_data_point dimension)
xn2, yn2 = int(num_days * 4), 200 # (The number of mission days multiplied by 4 dives per day (on average), 1000m / 5m per dive = 200 points)
xmin2, xmax2 = oxy_time_timestamps.min(), oxy_time_timestamps.max()
ymin2, ymax2 = 0, 1000
xgrid2 = np.linspace(xmin2, xmax2, xn2)
ygrid2 = np.linspace(ymin2, ymax2, yn2)
Xgrid2, Ygrid2 = np.meshgrid(xgrid2, ygrid2)

oxy_interp = griddata((oxy_time_timestamps, aa4831_depth), oxygen.values.flatten(), (Xgrid2, Ygrid2), method='linear')

# Check if interpolation was successful
if oxy_interp is None:
    print('Interpolation failed for oxygen')
else:
    print('Interpolation successful for oxygen')

sci_variables = {
    'oxygen': oxygen
}

for var_name, var in sci_variables.items():
    print(f'Processing {var_name} data....')

    var_directory = figures_folder + f'\{var_name}'
    os.makedirs(var_directory, exist_ok=True)  # exist_ok=True prevents errors if the folder already exists

    # Raw Scatter Plot
    plt.figure(figsize=(10, 5), dpi=300)
    plt.scatter(aa4831_time, aa4831_depth, c=var, cmap=cmocean.cm.oxy)
    plt.colorbar(label=f'{var.units}')
    plt.clim(0,300)
    plt.gca().invert_yaxis()
    plt.title(f'{time_coverage_start} - {time_coverage_end}')
    plt.xlabel('Time')
    plt.ylabel('Depth (m)')
    plt.ylim(1000,0)
    plt.grid(alpha=0.5)
    plt.savefig(f'{var_directory}/raw_{var_name}_mission.png')
    plt.close()
    print(f'raw_{var_name}_mission.png created')

    # Gridded Scatter Plot
    plt.figure(figsize=(10, 5), dpi=300)
    plt.scatter(Xgrid2, Ygrid2, c=oxy_interp, cmap=cmocean.cm.oxy)
    plt.colorbar(label=f'{var.units}')
    plt.gca().invert_yaxis()
    plt.title(f'{time_coverage_start} - {time_coverage_end}')
    plt.xlabel('Time')
    plt.ylabel('Depth (m)')
    plt.clim(0,300)
    plt.ylim(1000,0)
    plt.grid(alpha=0.5)
    plt.savefig(f'{var_directory}/gridded_{var_name}_mission.png')
    plt.close()
    print(f'gridded_{var_name}_mission.png created')

    # Contour Plot
    levels = np.arange(0, 300, 50)
    plt.figure(figsize=(10, 5), dpi=300)
    contour = plt.contourf(Xgrid2, Ygrid2, oxy_interp, levels=levels, cmap=cmocean.cm.oxy)
    contour_lines = plt.contour(Xgrid2, Ygrid2, oxy_interp, levels=levels, colors='black', linewidths=0.5)
    plt.clabel(contour_lines, inline=True, fontsize=8, fmt='%1.1f')
    plt.colorbar(contour, label=f'{var.units}')
    plt.gca().invert_yaxis()
    plt.title(f'{time_coverage_start} - {time_coverage_end}')
    plt.xlabel('Time')
    plt.ylabel('Depth (m)')
    plt.ylim(1000,0)
    plt.grid(alpha=0.5)
    plt.savefig(f'{var_directory}/contour_{var_name}_mission.png')
    plt.close()
    print(f'contour_{var_name}_mission.png created')

print('Done!')



The mission lasted 44 days.
Interpolation successful for oxygen
Processing oxygen data....
raw_oxygen_mission.png created
gridded_oxygen_mission.png created
contour_oxygen_mission.png created
Done!
